# 20 · Silver: reference crosswalks

Layer: silver · Runs after: nothing (seed data)

**What:** the small lookup tables that make the sources speak the same language: states, state keywords and hazards.

**Why:** the sources share no key (profiling section 5). These crosswalks are the conforming rules. They are written as data, not buried in code, so every mapping is reviewable in one place and versioned in Git.

In [0]:
%run ./00_common

In [0]:
# xw_state: every spelling seen in any source -> one state code. abs_code gives a stable order for surrogate keys.
STATES = [  # code, name, type, ABS code
    ("NSW", "New South Wales", "State", 1), ("VIC", "Victoria", "State", 2), ("QLD", "Queensland", "State", 3),
    ("SA", "South Australia", "State", 4), ("WA", "Western Australia", "State", 5), ("TAS", "Tasmania", "State", 6),
    ("NT", "Northern Territory", "Territory", 7), ("ACT", "Australian Capital Territory", "Territory", 8)]
REGIONS = [("FNQ", "QLD", "Far North Queensland"), ("SEQ", "QLD", "South East Queensland"),
           ("SWQ", "QLD", "South West Queensland"), ("NQ", "QLD", "North Queensland"), ("CQ", "QLD", "Central Queensland")]

xw_rows = []
for code, name, _, _ in STATES:
    xw_rows += [(code, code, "state code"), (name.upper(), code, "state name")]
xw_rows += [(r, code, f"sub-state region ({desc})") for r, code, desc in REGIONS]
xw_state = spark.createDataFrame(xw_rows, "source_value string, state_code string, mapping_type string")
write(xw_state, "silver", "xw_state")

state_ref = spark.createDataFrame(STATES, "state_code string, state_name string, state_type string, abs_code int")
write(state_ref, "silver", "ref_state")

In [0]:
# xw_state_keyword: used only when ICA's State is blank (DQ2). Priority 1 = state name, 2 = city/region.
KEYWORDS = [("queensland", "QLD", 1), ("qld", "QLD", 1), ("far north", "QLD", 2), ("brisbane", "QLD", 2), ("townsville", "QLD", 2),
            ("cairns", "QLD", 2), ("new south wales", "NSW", 1), ("nsw", "NSW", 1), ("sydney", "NSW", 2), ("newcastle", "NSW", 2),
            ("victoria", "VIC", 1), ("victorian", "VIC", 1), ("melbourne", "VIC", 2), ("tasmania", "TAS", 1), ("hobart", "TAS", 2),
            ("western australia", "WA", 1), ("perth", "WA", 2), ("south australia", "SA", 1), ("adelaide", "SA", 2),
            ("northern territory", "NT", 1), ("darwin", "NT", 2), ("canberra", "ACT", 2)]
write(spark.createDataFrame(KEYWORDS, "keyword string, state_code string, priority int"), "silver", "xw_state_keyword")

In [0]:
# xw_hazard: ICA Type and disaster-events category -> one hazard group and family. Anything not listed is out of scope.
HAZARDS = [("ica", "Cyclone", "Cyclone", "Meteorological"), ("ica", "Storm", "Storm", "Meteorological"),
           ("ica", "Hail", "Hail", "Meteorological"), ("ica", "Flood", "Flood", "Hydrological"),
           ("ica", "Bushfire", "Bushfire", "Fire"), ("ica", "Earthquake", "Earthquake", "Geophysical"),
           ("disaster_events", "Cyclone", "Cyclone", "Meteorological"), ("disaster_events", "Severe Storm", "Storm", "Meteorological"),
           ("disaster_events", "Tornado", "Storm", "Meteorological"), ("disaster_events", "Hail", "Hail", "Meteorological"),
           ("disaster_events", "Flood", "Flood", "Hydrological"), ("disaster_events", "Bushfire", "Bushfire", "Fire"),
           ("disaster_events", "Earthquake", "Earthquake", "Geophysical")]
write(spark.createDataFrame(HAZARDS, "source string, source_value string, hazard_group string, hazard_family string"), "silver", "xw_hazard")

for t in ["xw_state", "ref_state", "xw_state_keyword", "xw_hazard"]:
    log_step("silver", f"seed_{t}", f"silver.{t}", read("silver", t).count(), read("silver", t).count(), 0, "reference data")

### Coverage check
Every distinct ICA `Type` should either map or be deliberately out of scope (`Man-made`).

In [0]:
display(read("bronze", "ica").groupBy(F.trim("type").alias("ica_type")).count()
        .join(read("silver", "xw_hazard").where("source = 'ica'"), F.col("ica_type") == F.col("source_value"), "left")
        .select("ica_type", "count", F.coalesce("hazard_group", F.lit("— out of scope")).alias("hazard_group")).orderBy("ica_type"))